In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
import warnings
import pandas_ta
warnings.filterwarnings("ignore")

# --- Get S&P 500 tickers ---
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
html = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}).text
sp500 = pd.read_html(html)[0]
sp500["Symbol"] = sp500["Symbol"].str.replace(".", "-", regex=False)
symbols_list = sp500["Symbol"].unique().tolist()

# --- Dates ---
end_date = pd.to_datetime("today").normalize() - pd.Timedelta(days=1)  # yesterday
start_date = end_date - pd.DateOffset(years=8)

# --- Download ---
df = yf.download(
    tickers=symbols_list,
    start=start_date,
    end=end_date + pd.Timedelta(days=1),  # include end_date
    group_by="column",
    threads=True,
    auto_adjust=False
)

# If yfinance adds extra level (like "Price"), drop it
if df.columns.nlevels > 2:
    df = df.droplevel(0, axis=1)

# Ensure (field, ticker) order
fields = {"Open","High","Low","Close","Adj Close","Volume"}
lvl0_is_field = set(df.columns.get_level_values(0)) & fields == fields
if not lvl0_is_field:
    df = df.swaplevel(0, 1, axis=1)

df.columns = df.columns.set_names(["field","ticker"])

# --- Long form ---
df_long = df.stack("ticker").reset_index()
df_long.columns = ["date","ticker","open","high","low","close","adj_close","volume"]

print("Shape:", df_long.shape)
print(df_long.tail(20))   # shows the latest ~20 rows


In [ ]:
import pandas_ta as ta  # already imported

df = df_long.copy()

# --- 1. Clean data ---
req = ["open","high","low","adj_close"]
df = df.dropna(subset=req)
df = df[(df[req] > 0).all(axis=1)]

# --- 2. Garman–Klass variance & volatility (daily) ---
df["garman_klass_var"] = (
    0.5 * (np.log(df["high"] / df["low"]))**2
    - (2*np.log(2) - 1) * (np.log(df["adj_close"] / df["open"]))**2
)
df["garman_klass_vol"] = np.sqrt(df["garman_klass_var"])

# --- 3. RSI (20-day) ---
df["rsi"] = df.groupby("ticker", group_keys=False)["adj_close"].apply(
    lambda x: ta.rsi(x, length=20)
)

# --- 4. Bollinger Bands (20-day, std=2) ---
bb = df.groupby("ticker", group_keys=False).apply(
    lambda g: ta.bbands(g["adj_close"], length=20)[["BBL_20_2.0","BBM_20_2.0","BBU_20_2.0"]]
)

# join with df
df = df.join(bb)
df = df.rename(columns={
    "BBL_20_2.0": "bb_low",
    "BBM_20_2.0": "bb_mid",
    "BBU_20_2.0": "bb_high"
})

def compute_atr(stock_data):
    return ta.atr(
        high=stock_data['high'],
        low=stock_data['low'],
        close=stock_data['close'],
        length=14
    )

df["atr"] = df.groupby("ticker", group_keys=False).apply(compute_atr)

def compute_macd(close):
    macd_df = ta.macd(close=close, fast=12, slow=26, signal=9)
    return macd_df[["MACD_12_26_9", "MACDs_12_26_9", "MACDh_12_26_9"]]

macd = df.groupby("ticker", group_keys=False)["adj_close"].apply(compute_macd)

# join with df
df = df.join(macd)

df["dollar_volume"] = df["adj_close"] * df["volume"]

# --- 5. Sort & peek ---
df = df.sort_values(["date", "ticker"])
print(df[[
    "date","ticker","adj_close","open","high","low","close","volume",
    "garman_klass_vol","rsi","bb_low","bb_mid","bb_high","atr",
    "MACD_12_26_9","MACDs_12_26_9","MACDh_12_26_9","dollar_volume"
]].head(10))


In [ ]:
drop_cols = ['date','ticker','dollar_volume','volume','open','high','low','close']
last_cols = [c for c in df.columns if c not in drop_cols]

# set (date, ticker) as index ONCE
df_idx = df.set_index(["date","ticker"])

# 1. Indicators (last value of each month)
df_indicators_monthly = (
    df_idx[last_cols]                          # select indicators only
      .groupby("ticker")
      .resample("M", level="date").last()
      .reset_index()
)

# 2. Dollar Volume (mean or sum across month)
df_dv_monthly = (
    df_idx["dollar_volume"]
      .unstack("ticker")
      .resample("M").mean()   # or .sum()
      .stack("ticker")
      .to_frame("dollar_volume")
      .reset_index()
)

# 3. Merge
df_monthly = pd.merge(
    df_indicators_monthly,
    df_dv_monthly,
    on=["date","ticker"],
    how="inner"
)

print(df_monthly.head(20))



In [ ]:
# --- 1. Rolling 10-day average dollar volume ---
rolling_dv = (
    df.set_index(["date","ticker"])["dollar_volume"]   # pick series
      .unstack("ticker")                               # wide: tickers as cols
      .rolling(10).mean()                              # 10-day rolling mean
      .stack("ticker")                                 # back to long
      .to_frame("dollar_volume_rolling10")
      .reset_index()
)

# --- 2. Rank tickers by rolling dollar volume within each date ---
rolling_dv["dollar_vol_rank"] = (
    rolling_dv.groupby("date")["dollar_volume_rolling10"]
              .rank(ascending=False, method="first")   # highest volume = rank 1
)

top150_clean = top150.drop(columns=["dollar_volume_rolling10", "dollar_vol_rank"])

print(top150_clean.head(150))



In [ ]:
def add_momentum_features(g, lags=[1,2,3,6,9,12], outlier_cutoff=0.005):
    for lag in lags:
        g[f'return_{lag}m'] = (
            g['adj_close']
              .pct_change(lag)
              .pipe(lambda x: x.clip(
                  lower=x.quantile(outlier_cutoff),
                  upper=x.quantile(1-outlier_cutoff)
              ))
              .add(1)
              .pow(1/lag)
              .sub(1)
        )
    # drop rows where returns are NaN (first few rows per ticker)
    return g.dropna(subset=[f'return_{lag}m' for lag in lags])

# Apply per ticker
df = (
    df.groupby("ticker", group_keys=False)
      .apply(add_momentum_features)
      .reset_index(drop=True)
)

# Preview
print(df[["date","ticker","adj_close","return_1m","return_3m","return_6m","return_12m"]].head(150))


In [ ]:
# --- Get Fama-French 5-Factor data ---
factor_data = web.DataReader('F-F_Research_Data_5_Factors_2x3',
                             'famafrench',
                             start=2010)[0]

# Drop RF, convert % to decimal
factor_data = factor_data.drop(columns="RF").div(100)

# Convert PeriodIndex to Timestamp for merging
factor_data.index = factor_data.index.to_timestamp()
factor_data.index.name = "date"

# Reset index so 'date' is a column for merge
factor_data = factor_data.reset_index()

# --- Merge with your stock data ---
# (df must have ['date','ticker','return_1m'])
merged = pd.merge(
    df.reset_index()[["date","ticker","return_1m","rsi"]],  # stock data
    factor_data,                                      # factor data
    on="date",
    how="inner"
)

# Set MultiIndex for later regression
merged = merged.set_index(["date","ticker"])

print(merged.head(20))


In [ ]:
observations = df.groupby("ticker").size()

# Keep only tickers with >= 10 months of history
valid_stocks = observations[observations >= 10]

# Print the list of valid stocks
print(valid_stocks)

# Or just the tickers
print(valid_stocks.index.tolist())

In [ ]:
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS

# Run rolling regressions per ticker
def rolling_ff_betas(group, window=24):
    # Drop rows with missing data
    group = group.dropna(subset=["return_1m","Mkt-RF","SMB","HML","RMW","CMA"])
    if len(group) < window:   # not enough history
        return pd.DataFrame(index=group.index, 
                            columns=["Mkt-RF","SMB","HML","RMW","CMA"])
    
    y = group["return_1m"]
    X = sm.add_constant(group[["Mkt-RF","SMB","HML","RMW","CMA"]])
    
    model = RollingOLS(endog=y, exog=X, window=window, min_nobs=len(X.columns)+1)
    res = model.fit()
    
    betas = res.params.drop("const", axis=1)   # drop intercept
    return betas

# Apply per ticker on the merged dataset
betas = (
    merged.groupby("ticker", group_keys=False)
          .apply(rolling_ff_betas, window=24)
)

print(betas.head(200))



In [ ]:
factors = ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']

# 1. Rename betas to avoid overlap
betas = betas.rename(columns={f: f"beta_{f}" for f in factors})

# 2. Join lagged betas to dataset
merged = merged.join(betas.groupby('ticker').shift())

# 3. Fill missing betas with each ticker’s mean
beta_cols = [f"beta_{f}" for f in factors]
merged.loc[:, beta_cols] = (
    merged.groupby('ticker', group_keys=False)[beta_cols]
          .apply(lambda x: x.fillna(x.mean()))
)

# 4. Drop adj_close if exists
if 'adj_close' in merged.columns:
    merged = merged.drop('adj_close', axis=1)

# 5. Drop leftover NaNs
merged = merged.dropna()

merged.info()


In [ ]:
from sklearn.cluster import KMeans

# Features to use in clustering
cluster_features = [
    "return_1m", 
    "beta_Mkt-RF", "beta_SMB", "beta_HML", "beta_RMW", "beta_CMA", "rsi"
]

# Drop existing cluster column if it exists
if "cluster" in merged.columns:
    merged = merged.drop("cluster", axis=1)

# Function to cluster per month
def get_clusters(df):
    kmeans = KMeans(
        n_clusters=4,
        random_state=0,
        init="k-means++"
    )
    df = df.copy()
    df["cluster"] = kmeans.fit_predict(df[cluster_features])
    return df

# Apply clustering per month
merged = (
    merged.dropna(subset=cluster_features)
          .groupby("date", group_keys=False)
          .apply(get_clusters)
)

print(merged.head(20))



In [ ]:
import matplotlib.pyplot as plt

def plot_clusters(data, x_col="return_1m", y_col="beta_Mkt-RF"):
    """
    Plot KMeans clusters in 2D using two chosen features.
    
    Parameters:
    -----------
    data : DataFrame
        Your dataframe containing features + 'cluster'
    x_col : str
        Column name for X-axis (default = 'return_1m')
    y_col : str
        Column name for Y-axis (default = 'beta_Mkt-RF')
    """
    plt.figure(figsize=(8,6))

    for cluster_id, color in zip(range(data["cluster"].nunique()), 
                                 ["red", "green", "blue", "black"]):
        cluster = data[data["cluster"] == cluster_id]
        plt.scatter(cluster[x_col], cluster[y_col], 
                    label=f"Cluster {cluster_id}", 
                    alpha=0.6, c=color)

    plt.xlabel(x_col)
    plt.ylabel(y_col)
    plt.title(f"Clusters based on {x_col} vs {y_col}")
    plt.legend()
    plt.show()


In [ ]:
plt.style.use('ggplot')

# Loop over each unique date in your panel data
for i in merged.index.get_level_values('date').unique().tolist():
    
    # Take all tickers for that month
    g = merged.xs(i, level="date")
    
    # Title = which month we’re plotting
    plt.title(f"Date {i.date()}")   # .date() makes it cleaner (YYYY-MM-DD)
    
    # Call your cluster plotting function
    plot_clusters(g)


In [ ]:
import numpy as np

# RSI values to guide clusters
target_rsi_values = [30, 45, 55, 70]

# Initialize centroids: (4 clusters × 7 features)
initial_centroids = np.zeros((len(target_rsi_values), len(cluster_features)))

# Place RSI values in the last column (index 6, since RSI is the 7th feature)
initial_centroids[:, -1] = target_rsi_values
print("Initial centroids (with RSI guidance):")
print(initial_centroids)

In [ ]:
# 1. Filter only Cluster 3 stocks (your momentum hypothesis)
filtered_df = merged[merged["cluster"] == 3].copy()

# 2. Reset ticker level, shift dates forward by 1 day (to avoid lookahead bias)
filtered_df = filtered_df.reset_index(level="ticker")
filtered_df.index = filtered_df.index + pd.DateOffset(1)

# 3. Restore MultiIndex (date, ticker)
filtered_df = filtered_df.reset_index().set_index(["date", "ticker"])

# 4. Extract all unique dates
dates = filtered_df.index.get_level_values("date").unique().tolist()

# 5. Create dictionary: rebalance_date → list of tickers
fixed_dates = {
    d.strftime("%Y-%m-%d"): filtered_df.xs(d, level=0).index.tolist()
    for d in dates
}

# Preview first few dates + tickers
for k, v in list(fixed_dates.items())[:100]:
    print(f"{k}: {v}")


In [ ]:
# 1. Extract all unique tickers from your panel data
stocks = merged.index.get_level_values("ticker").unique().tolist()

# 2. Define start and end dates (add 12 months before the first date for lookback window)
start_date = merged.index.get_level_values("date").min() - pd.DateOffset(months=12)
end_date   = merged.index.get_level_values("date").max()

# 3. Download adjusted close price data only (easier to work with for returns)
new_df = yf.download(
    tickers=stocks,
    start=start_date,
    end=end_date,
    auto_adjust=True,     # adjusts OHLC automatically
    progress=False
)["Close"]                # we only keep the adjusted 'Close'

# 4. Ensure columns are just ticker symbols
new_df = new_df.loc[:, ~new_df.columns.duplicated()]  

print("Shape:", new_df.shape)
print(new_df.head())

In [ ]:
print(optimization_df.head())
print(optimization_df.isna().sum())


In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
from pypfopt.expected_returns import mean_historical_return
from pypfopt.risk_models import CovarianceShrinkage
from pypfopt.efficient_frontier import EfficientFrontier

# =========================================================
# 1. Download and clean price data
# =========================================================
# Extract tickers from your merged dataset
stocks = merged.index.get_level_values("ticker").unique().tolist()

# Define start/end dates
start_date = merged.index.get_level_values("date").min() - pd.DateOffset(months=12)
end_date   = merged.index.get_level_values("date").max()

# Download full OHLCV
new_df = yf.download(stocks, start=start_date, end=end_date, group_by="ticker", auto_adjust=False)

# ✅ Fix: Extract Adjusted Close into a clean DataFrame
adj_close_df = new_df.xs("Adj Close", axis=1, level=1)

# Compute daily log returns
returns_dataframe = np.log(adj_close_df).diff()

# =========================================================
# 2. Helper function for Max Sharpe optimization
# =========================================================
def optimize_weights(prices, lower_bound=0.01):
    # Compute mean returns and covariance matrix
    mu = mean_historical_return(prices, frequency=252)   # annualized mean return
    S = CovarianceShrinkage(prices).ledoit_wolf()        # shrinkage covariance for stability

    # Efficient Frontier
    ef = EfficientFrontier(mu, S, weight_bounds=(lower_bound, 1))
    weights = ef.max_sharpe()
    cleaned_weights = ef.clean_weights()

    return cleaned_weights

# =========================================================
# 3. Portfolio backtest loop
# =========================================================
portfolio_df = pd.DataFrame()

for start_date in fixed_dates.keys():
    try:
        # Holding period = month
        end_date = (pd.to_datetime(start_date) + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d')

        # Stocks in the chosen cluster
        cols = fixed_dates[start_date]

        # Lookback window = 12 months before rebalance
        optimization_start_date = (pd.to_datetime(start_date) - pd.DateOffset(months=12)).strftime('%Y-%m-%d')
        optimization_end_date = (pd.to_datetime(start_date) - pd.DateOffset(days=1)).strftime('%Y-%m-%d')
        optimization_df = adj_close_df.loc[optimization_start_date:optimization_end_date, cols]

        # Skip if insufficient data
        if optimization_df.shape[1] < 2 or optimization_df.dropna().shape[0] < 50:
            print(f"Skipping {start_date} (not enough data)")
            continue

        # Try optimization
        try:
            weights = optimize_weights(prices=optimization_df,
                                       lower_bound=round(1 / (len(optimization_df.columns) * 2), 3))
            weights = pd.Series(weights)
        except Exception as e:
            print(f"⚠️ Max Sharpe failed for {start_date}, using Equal Weights ({e})")
            weights = pd.Series([1/len(optimization_df.columns)]*len(optimization_df.columns),
                                index=optimization_df.columns)

        # Compute strategy returns for this month
        temp_df = returns_dataframe.loc[start_date:end_date, cols].copy()
        temp_df = temp_df.mul(weights, axis=1).sum(axis=1).to_frame("Strategy Return")

        portfolio_df = pd.concat([portfolio_df, temp_df])

    except Exception as e:
        print(f"❌ Error at {start_date}: {e}")

# =========================================================
# 4. Clean-up and cumulative return
# =========================================================
portfolio_df = portfolio_df[~portfolio_df.index.duplicated(keep="first")]
portfolio_df["Cumulative Return"] = (1 + portfolio_df["Strategy Return"]).cumprod()

# =========================================================
# 5. Benchmark (SPY) comparison
# =========================================================
benchmark = yf.download("SPY", start=portfolio_df.index.min(), end=portfolio_df.index.max(), auto_adjust=True)
benchmark_returns = np.log(benchmark["Close"]).diff()
benchmark_cum = (1 + benchmark_returns).cumprod()

portfolio_df["SPY Return"] = benchmark_returns.reindex(portfolio_df.index)
portfolio_df["SPY Cumulative"] = benchmark_cum.reindex(portfolio_df.index)

# =========================================================
# 6. Results
# =========================================================
print(portfolio_df.tail(200))
print("\nFull shape:", portfolio_df.shape)
